
## 1. Instalación de dependencias

Ejecuta esta celda solo si necesitas instalar las bibliotecas en tu equipo.


In [ ]:
!pip install torch torchvision matplotlib


# Importaciones y configuración general


In [ ]:

import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)



In [ ]:

possible_roots = [
    Path('Workspace'),
    Path('images'),
    Path('images/Workspace'),
    Path('./images/Workspace'),
]

DATA_ROOT = None
for root in possible_roots:
    if (root / 'seg_train').exists() and (root / 'seg_test').exists():
        DATA_ROOT = root
        break

TRAIN_DIR = DATA_ROOT / 'seg_train'
TEST_DIR = DATA_ROOT / 'seg_test'

print('DATA_ROOT =', DATA_ROOT.resolve())
print('TRAIN_DIR =', TRAIN_DIR.resolve())
print('TEST_DIR  =', TEST_DIR.resolve())



# Transformaciones

In [ ]:

IMAGE_SIZE = 150
BATCH_SIZE = 4
NUM_WORKERS = 2

train_transforms = transforms.Compose([
    transforms.RandomRotation(degrees=20),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomResizedCrop(size=(IMAGE_SIZE, IMAGE_SIZE), scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.3),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

original_visual_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])



# Cargar el dataset con `ImageFolder`


In [ ]:

train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=eval_transforms)
train_dataset_original = datasets.ImageFolder(root=TRAIN_DIR, transform=original_visual_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
visual_original_loader = DataLoader(train_dataset_original, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
visual_aug_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

class_names = train_dataset.classes
print('Clases:', class_names)
print('Número de clases:', len(class_names))
print('Imágenes de entrenamiento:', len(train_dataset))
print('Imágenes de prueba:', len(test_dataset))



# Visualizar imágenes originales y transformadas


In [ ]:

def denormalize(img_tensor, mean, std):
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return img_tensor * std + mean


def show_batch(images, labels, classes, title, normalized=False):
    fig, axes = plt.subplots(1, len(images), figsize=(16, 4))
    fig.suptitle(title, fontsize=14)
    for i, ax in enumerate(axes):
        img = images[i].cpu()
        if normalized:
            img = denormalize(img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        img = torch.clamp(img, 0, 1)
        ax.imshow(img.permute(1, 2, 0))
        ax.set_title(classes[labels[i]])
        ax.axis('off')
    plt.tight_layout()
    plt.show()

orig_images, orig_labels = next(iter(visual_original_loader))
aug_images, aug_labels = next(iter(visual_aug_loader))

show_batch(orig_images, orig_labels.tolist(), class_names, 'Lote original', normalized=False)
show_batch(aug_images, aug_labels.tolist(), class_names, 'Lote transformado', normalized=True)



# Definición de la CNN


In [ ]:

class IntelCNN(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        with torch.no_grad():
            sample = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
            flattened_size = self.features(sample).view(1, -1).shape[1]

        self.classifier = nn.Sequential(
            nn.Linear(flattened_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model = IntelCNN(num_classes=len(class_names)).to(DEVICE)
print(model)



# Configurar función de pérdida y optimizador

In [ ]:

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-5)
EPOCHS = 30



# Entrenamiento del modelo


In [ ]:

train_losses = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_loss)
    print(f'Época [{epoch+1}/{EPOCHS}] - Pérdida: {epoch_loss:.4f}')



# Curva de pérdida


In [ ]:

plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS + 1), train_losses, marker='o')
plt.title('Pérdida de entrenamiento por época')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.grid(True)
plt.show()


# Evaluación del modelo con imágenes sin aumento


In [ ]:

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Precisión final en test: {accuracy:.2f}%')



# Guardar el modelo entrenado


In [ ]:

MODEL_PATH = 'intel_cnn_model.pth'
torch.save(model.state_dict(), MODEL_PATH)
print(f'Modelo guardado en: {MODEL_PATH}')
